# **Symbolic Regression: Demonstration**

In [1]:
import numpy as np
import pandas as pd
from pystar import SymbolicRegressionModel, setup_logger
from pystar.solution_approach.miqp_solver import PoolAndPrune

setup_logger()

pystar: INFO: ================================================================================
pystar: INFO: Configuring a new pystar logger: [2026-11-03 10:26:36.286616] 


#### **1. Langmuir Isotherm**
Since ALAMO does not support inverse of affine functions as basis functions, it cannot find the correct expression.

$$ \theta = \frac{K_L \cdot P}{1 + K_L \cdot P}$$

In [2]:
np.random.seed(42)
data = pd.DataFrame(
    np.random.uniform(low=0.1, high=10, size=(10, 1)), columns=["P"]
)
kl = 1.5
data["theta"] = kl * data["P"] / (1 + kl * data["P"])

In [3]:
# Solve using the big-m transformation
m = SymbolicRegressionModel(
    data=data,
    input_columns=["P"],
    output_column="theta",
    tree_depth=4,
    operators=["sum", "diff", "mult", "div", "square", "sqrt"],
    var_bounds=(-100, 100),
    constant_bounds=(-100, 100),
    model_type="hull",
)
m.add_objective(objective_type="sse")

m.add_constant_operation_cuts()
m.add_implication_cuts()
m.add_associative_operation_cuts()
m.add_inverse_function_composition_cuts()
m.add_same_operand_operation_cuts()
m.add_similar_operation_cuts()

pystar: INFO: Using ['square', 'sqrt'] unary operators.
pystar: INFO: Using ['sum', 'diff', 'mult', 'div'] binary operators.
pystar: INFO: Fixing select_operator[2 n + 1, cst] variables to zero.


In [4]:
solver = PoolAndPrune(model=m, pool_size=20, overall_time_limit=3600)
solver.solve()

pystar: INFO: Created a clone of the MINLP; nonconvex constraints in each disjunct, if present, are relaxed.
Set parameter MIPGap to value 0
Set parameter PoolSearchMode to value 2
Set parameter PoolSolutions to value 20
pystar: INFO: Created a clone of the MINLP; this model is used for NLP solves.
Set parameter MIPGap to value 0
Set parameter TimeLimit to value 100
pystar: INFO: Solver object construction is complete.
pystar: INFO: Expr. Num.    Lower bound    Incumbent Sol./Status      Upper bound      Time
pystar: INFO: ==========    ===========    =====================      ===========    =======
Set parameter TimeLimit to value 3599.9989173
pystar: WARNING: Fixing c_10 in (c_12 - x1)*(c_8 - x1)/(c_10*c_14*x1**2) to 1
pystar: WARNING: Fixing c_14 in (c_12 - x1)*(c_8 - x1)/(c_10*c_14*x1**2) to 1
pystar: INFO:          1        0.00000        0.01007/    optimal        0.01007          6
pystar: WARNING: Fixing c_10 in (c_12 - x1)*(c_8 + x1)/(c_10*c_14*x1**2) to 1
pystar: WARNING: Fi

In [5]:
sol_pool = solver.expression_pool

In [6]:
sol_pool

,Lower Bound,Relaxation Time,Expression,NLP Status,Upper Bound,Total Time
1,0.0,0.315,1.0*(0.660157185882006 - x1)*(x1 - 6.602853140...,optimal,8.665753,8.615469
2,0.0,0.315,0.352397435848623/x1**2,optimal,6.39113,8.678817
3,0.0,0.315,0.129216451258158*x1,optimal,0.953712,10.524988
4,0.0,0.315,sqrt(x1),optimal,21.787035,10.586617
5,0.0,0.315,-x1**(1/4),optimal,53.317293,10.646763
...,...,...,...,...,...,...
458,0.0,0.487,-0.0126498016844196*x1**3 + x1,optimal,28.048505,858.100805
459,0.0,0.487,0.830318007057825/x1,optimal,4.872652,859.101644
460,0.0,0.519,1.0*(x1 + 33.2575620782324)/(52.1956316197123 ...,optimal,0.043405,863.199297
461,0.0,0.519,(5.58430244653653*x1 + 3.97126076912002e-7)/(5...,optimal,0.0,868.972218


In [12]:
# Expressions that were not solved to optimality
z = sol_pool[sol_pool["NLP Status"] == "feasible"]
for expr in z["Expression"]:
    print(expr.sympy_expression)

(c_12 + c_14*x1 + x1)*(c_10*x1 + c_8 + x1)
(c_12 + c_14*x1 - x1)*(c_10*x1 + c_8 + x1)
-(c_12 + x1)/(c_14*x1) + (c_8 + x1)/(c_10*x1)
(c_10*x1 + c_8 + x1)*(c_12*x1 + c_14 - x1)
(c_12*x1 + c_14/x1)*(c_10*x1 + c_8 + x1)


In [10]:
best_solution = solver.best_expression_tree
print(best_solution)

(5.58430244653653*x1 + 3.97126076912002e-7)/(5.58430239918229*x1 + 3.72286903053289)
